# Моделирование. Проведение экспериментов. Подготовка пайплайнов обработки данных и построения модели.

# Импорты и настройки

In [57]:
# Стандартная библиотека
import os
import sys
import warnings
import joblib

# Работа с данными
import numpy as np
import pandas as pd

# ML
import lightgbm as lgb
import mlflow

from scipy import sparse
from scipy.sparse import csr_matrix

from implicit.als import AlternatingLeastSquares

# Настройки
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)
pd.options.display.float_format = "{:.6f}".format

# Пути проекта
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from src.config import TRAIN_PATH, DATA_DIR, RAW_DIR

MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
os.makedirs(MODELS_DIR, exist_ok=True)

# Проверка путей
print("PROJECT_ROOT:", PROJECT_ROOT)
print("TRAIN_PATH exists:", os.path.exists(TRAIN_PATH))
print("DATA_DIR exists:", os.path.exists(DATA_DIR))
print("RAW_DIR exists:", os.path.exists(RAW_DIR))
print("MODELS_DIR exists:", os.path.exists(MODELS_DIR))

PROJECT_ROOT: /Users/andrejmoldovan/Desktop/Магистратура РАНХИГС и Яндекс/4 семестр/Рекомендации банковских продуктов/bank-product-recs
TRAIN_PATH exists: True
DATA_DIR exists: True
RAW_DIR exists: True
MODELS_DIR exists: True


# Загрузка данных

In [7]:
df = pd.read_csv(TRAIN_PATH, low_memory=False)
print(df.shape)
df.head()

(13647309, 48)


,fecha_dato,ncodpers,ind_empleado,pais_residencia,sexo,age,fecha_alta,ind_nuevo,antiguedad,indrel,ult_fec_cli_1t,indrel_1mes,tiprel_1mes,indresi,indext,conyuemp,canal_entrada,indfall,tipodom,cod_prov,nomprov,ind_actividad_cliente,renta,segmento,ind_ahor_fin_ult1,ind_aval_fin_ult1,ind_cco_fin_ult1,ind_cder_fin_ult1,ind_cno_fin_ult1,ind_ctju_fin_ult1,ind_ctma_fin_ult1,ind_ctop_fin_ult1,ind_ctpp_fin_ult1,ind_deco_fin_ult1,ind_deme_fin_ult1,ind_dela_fin_ult1,ind_ecue_fin_ult1,ind_fond_fin_ult1,ind_hip_fin_ult1,ind_plan_fin_ult1,ind_pres_fin_ult1,ind_reca_fin_ult1,ind_tjcr_fin_ult1,ind_valo_fin_ult1,ind_viv_fin_ult1,ind_nomina_ult1,ind_nom_pens_ult1,ind_recibo_ult1
0,2015-01-28,1375586,N,ES,H,35,2015-01-12,0,6,1,NaN,1.0,A,S,N,NaN,KHL,N,1,29,MALAGA,1,"87,218",02 - PARTICULARES,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,2015-01-28,1050611,N,ES,V,23,2012-08-10,0,35,1,NaN,1,I,S,S,NaN,KHE,N,1,13,CIUDAD REAL,0,"35,549",03 - UNIVERSITARIO,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,2015-01-28,1050612,N,ES,V,23,2012-08-10,0,35,1,NaN,1,I,S,N,NaN,KHE,N,1,13,CIUDAD REAL,0,"122,179",03 - UNIVERSITARIO,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,2015-01-28,1050613,N,ES,H,22,2012-08-10,0,35,1,NaN,1,I,S,N,NaN,KHD,N,1,50,ZARAGOZA,0,"119,776",03 - UNIVERSITARIO,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,2015-01-28,1050614,N,ES,V,23,2012-08-10,0,35,1,NaN,1,A,S,N,NaN,KHE,N,1,50,ZARAGOZA,1,NaN,03 - UNIVERSITARIO,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


# Базовая предобработка

In [8]:
# Даты
date_cols = ["fecha_dato", "fecha_alta", "ult_fec_cli_1t"]

for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")


# Продуктовые признаки (таргет/фичи)
product_cols = [c for c in df.columns if c.endswith("_ult1")]

for col in product_cols:
    df[col] = (
        pd.to_numeric(df[col], errors="coerce")
        .fillna(0)
        .clip(0, 1)   # защита от мусора
        .astype(np.uint8)
    )


# Числовые признаки
num_cols = [
    "age",
    "antiguedad",
    "renta",
    "ind_nuevo",
    "indrel",
    "tipodom",
    "cod_prov",
    "ind_actividad_cliente"
]

for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")


# Дополнительная очистка числовых
# возраст
if "age" in df.columns:
    df["age"] = df["age"].clip(18, 100)

# стаж
if "antiguedad" in df.columns:
    df["antiguedad"] = df["antiguedad"].clip(0, 500)

# доход (очень важный признак)
if "renta" in df.columns:
    df["renta"] = df["renta"].clip(0, df["renta"].quantile(0.99))
    df["renta_log"] = np.log1p(df["renta"])


# Категориальные признаки
cat_cols = [
    "ind_empleado",
    "pais_residencia",
    "sexo",
    "indrel_1mes",
    "tiprel_1mes",
    "indresi",
    "indext",
    "canal_entrada",
    "indfall",
    "nomprov",
    "segmento"
]

for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].astype("category")


# Сортировка по времени
df = df.sort_values(["ncodpers", "fecha_dato"]).reset_index(drop=True)

# Формирование таргета

In [9]:
# сортировка по клиенту и времени
df = df.sort_values(["ncodpers", "fecha_dato"]).copy()

# предыдущий продуктовый портфель клиента
prev_products = df.groupby("ncodpers")[product_cols].shift(1)

# признаки предыдущего состояния
prev_product_cols = [f"prev_{col}" for col in product_cols]
prev_products_for_features = prev_products.fillna(0).astype(np.uint8).copy()
prev_products_for_features.columns = prev_product_cols

# таргет: только новые продукты
target_df = (df[product_cols] - prev_products.fillna(0)).clip(lower=0).astype(np.uint8)
target_cols = [f"target_{col}" for col in product_cols]
target_df.columns = target_cols

# объединяем
df = pd.concat([df, prev_products_for_features, target_df], axis=1)

# удаляем первое наблюдение клиента, у которого нет истории
df["row_num_per_client"] = df.groupby("ncodpers").cumcount()
df = df[df["row_num_per_client"] > 0].copy()

print(df.shape)
print("Количество target-признаков:", len(target_cols))
print("Количество prev-признаков:", len(prev_product_cols))
print("NaN в таргете:", df[target_cols].isna().sum().sum())

(12690664, 98)
Количество target-признаков: 24
Количество prev-признаков: 24
NaN в таргете: 0


In [10]:
# распределение таргета
df["new_products"] = df[target_cols].sum(axis=1)

print(df["new_products"].describe())
print(df["new_products"].value_counts(normalize=True).sort_index().head(10))

count   12,690,664
mean             0
std              0
min              0
25%              0
50%              0
75%              0
max              7
Name: new_products, dtype: float64
new_products
0   1
1   0
2   0
3   0
4   0
5   0
6   0
7   0
Name: proportion, dtype: float64


In [11]:
# самые частые новые продукты
display(df[target_cols].sum().sort_values(ascending=False).head(10))

target_ind_recibo_ult1      153205
target_ind_nom_pens_ult1     84768
target_ind_nomina_ult1       73800
target_ind_cco_fin_ult1      69997
target_ind_tjcr_fin_ult1     69311
target_ind_cno_fin_ult1      37187
target_ind_ecue_fin_ult1     26378
target_ind_dela_fin_ult1     12707
target_ind_reca_fin_ult1      9238
target_ind_ctma_fin_ult1      7002
dtype: uint64

# Feature engineering

In [12]:
# агрегированные признаки
df["num_products"] = df[product_cols].sum(axis=1).astype(np.uint8)

df["tenure_days"] = (df["fecha_dato"] - df["fecha_alta"]).dt.days
df["tenure_days"] = df["tenure_days"].clip(lower=0)
df["tenure_days"] = df["tenure_days"].fillna(df["tenure_days"].median())

df["month"] = df["fecha_dato"].dt.month.astype(np.uint8)

# списки признаков
num_features = [
    "age",
    "antiguedad",
    "renta",
    "renta_log",
    "ind_nuevo",
    "indrel",
    "tipodom",
    "cod_prov",
    "ind_actividad_cliente",
    "num_products",
    "tenure_days",
    "month",
]

cat_features = [
    "ind_empleado",
    "pais_residencia",
    "sexo",
    "indrel_1mes",
    "tiprel_1mes",
    "indresi",
    "indext",
    "canal_entrada",
    "indfall",
    "nomprov",
    "segmento",
]

num_features = [c for c in num_features if c in df.columns]
cat_features = [c for c in cat_features if c in df.columns]

# используем только prev_* как продуктовые признаки
feature_cols = num_features + cat_features + prev_product_cols

print("Количество числовых признаков:", len(num_features))
print("Количество категориальных признаков:", len(cat_features))
print("Количество prev-признаков:", len(prev_product_cols))
print("Итого признаков:", len(feature_cols))

# проверка на leakage
leak_cols = [c for c in feature_cols if c in product_cols]
print("Leakage product_cols in feature_cols:", leak_cols)

Количество числовых признаков: 12
Количество категориальных признаков: 11
Количество prev-признаков: 24
Итого признаков: 47
Leakage product_cols in feature_cols: []


## Обработка категорий

### Для LightGBM можно закодировать категории через category.

In [13]:
# 1. Числовые признаки
num_features = [
    "age",
    "antiguedad",
    "renta",
    "renta_log",
    "ind_nuevo",
    "indrel",
    "tipodom",
    "cod_prov",
    "ind_actividad_cliente",
    "num_products",
    "tenure_days",
    "month",
]

num_features = [c for c in num_features if c in df.columns]

for col in num_features:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# базовая очистка
if "age" in df.columns:
    df["age"] = df["age"].clip(18, 100)

if "antiguedad" in df.columns:
    df["antiguedad"] = df["antiguedad"].clip(0, 500)

if "tenure_days" in df.columns:
    df["tenure_days"] = df["tenure_days"].clip(lower=0)

if "renta" in df.columns:
    upper_renta = df["renta"].quantile(0.99)
    df["renta"] = df["renta"].clip(lower=0, upper=upper_renta)

if "renta_log" not in df.columns and "renta" in df.columns:
    df["renta_log"] = np.log1p(df["renta"])

# медианная импутация
for col in num_features:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())


# 2. Категориальные признаки
cat_features = [
    "ind_empleado",
    "pais_residencia",
    "sexo",
    "indrel_1mes",
    "tiprel_1mes",
    "indresi",
    "indext",
    "canal_entrada",
    "indfall",
    "nomprov",
    "segmento",
]

cat_features = [c for c in cat_features if c in df.columns]

for col in cat_features:
    df[col] = df[col].astype(str).replace("nan", "UNKNOWN").astype("category")


# 3. Финальный список признаков

feature_cols = num_features + cat_features + prev_product_cols

print("Количество числовых признаков:", len(num_features))
print("Количество категориальных признаков:", len(cat_features))
print("Количество prev-признаков:", len(prev_product_cols))
print("Итого признаков:", len(feature_cols))

print("\nПример числовых признаков:")
print(num_features[:10])

print("\nПример категориальных признаков:")
print(cat_features[:10])

print("\nПример prev-признаков:")
print(prev_product_cols[:5])

Количество числовых признаков: 12
Количество категориальных признаков: 11
Количество prev-признаков: 24
Итого признаков: 47

Пример числовых признаков:
['age', 'antiguedad', 'renta', 'renta_log', 'ind_nuevo', 'indrel', 'tipodom', 'cod_prov', 'ind_actividad_cliente', 'num_products']

Пример категориальных признаков:
['ind_empleado', 'pais_residencia', 'sexo', 'indrel_1mes', 'tiprel_1mes', 'indresi', 'indext', 'canal_entrada', 'indfall', 'nomprov']

Пример prev-признаков:
['prev_ind_ahor_fin_ult1', 'prev_ind_aval_fin_ult1', 'prev_ind_cco_fin_ult1', 'prev_ind_cder_fin_ult1', 'prev_ind_cno_fin_ult1']


In [14]:
# функция оптимизации памяти
def reduce_memory_usage(df: pd.DataFrame) -> pd.DataFrame:
    start_mem = df.memory_usage(deep=True).sum() / 1024**2
    print(f"Memory before: {start_mem:.2f} MB")

    for col in df.columns:
        col_type = df[col].dtype

        # пропускаем даты
        if pd.api.types.is_datetime64_any_dtype(col):
            continue

        # числовые
        if pd.api.types.is_numeric_dtype(col):
            c_min = df[col].min()
            c_max = df[col].max()

            if pd.api.types.is_integer_dtype(col):
                if c_min >= 0:
                    if c_max < 255:
                        df[col] = df[col].astype(np.uint8)
                    elif c_max < 65535:
                        df[col] = df[col].astype(np.uint16)
                    else:
                        df[col] = df[col].astype(np.uint32)
                else:
                    df[col] = df[col].astype(np.int32)

            else:
                df[col] = df[col].astype(np.float32)

        # строки → category
        elif df[col].dtype == "object":
            num_unique = df[col].nunique()
            num_total = len(df[col])

            if num_unique / num_total < 0.5:
                df[col] = df[col].astype("category")

    end_mem = df.memory_usage(deep=True).sum() / 1024**2
    print(f"Memory after: {end_mem:.2f} MB")
    print(f"Reduced by: {(start_mem - end_mem) / start_mem * 100:.1f}%")

    return df

In [15]:
# оптимизируем память
df = reduce_memory_usage(df)

Memory before: 3074.15 MB
Memory after: 3074.15 MB
Reduced by: 0.0%


## Time-based split

In [16]:
train_df = df[df["fecha_dato"] < "2016-05-28"].copy()
valid_df = df[df["fecha_dato"] == "2016-05-28"].copy()

train_df["new_products"] = train_df[target_cols].sum(axis=1)
valid_df["new_products"] = valid_df[target_cols].sum(axis=1)

print("train shape:", train_df.shape)
print("valid shape:", valid_df.shape)
print("train share with purchases:", (train_df["new_products"] > 0).mean())
print("valid share with purchases:", (valid_df["new_products"] > 0).mean())

train shape: (11763904, 102)
valid shape: (926760, 102)
train share with purchases: 0.03570132840254392
valid share with purchases: 0.030122145970909404


### Почему разбиваем именно так?

* Это последний месяц в train-данных, 2016-05-28 — это последняя дата этого месяца.

* Мы моделируем будущее. Ближайшая дата к “будущему”

## Подготовка матриц признаков

In [18]:
X_train = train_df[feature_cols].copy()
X_valid = valid_df[feature_cols].copy()

for col in cat_features:
    X_train[col] = (
        X_train[col]
        .astype("category")
        .cat.add_categories(["UNKNOWN"])
        .fillna("UNKNOWN")
    )

    X_valid[col] = (
        X_valid[col]
        .astype("category")
        .cat.add_categories(["UNKNOWN"])
        .fillna("UNKNOWN")
    )

for col in num_features:
    X_train[col] = pd.to_numeric(X_train[col], errors="coerce")
    X_valid[col] = pd.to_numeric(X_valid[col], errors="coerce")

    median_value = X_train[col].median()
    X_train[col] = X_train[col].fillna(median_value)
    X_valid[col] = X_valid[col].fillna(median_value)

print(X_train.shape, X_valid.shape)

(11763904, 47) (926760, 47)


## Готовим матрицу user-item для ALS

In [50]:
BASE_DIR = os.getcwd()
MODELS_DIR = os.path.join(BASE_DIR, "models")
os.makedirs(MODELS_DIR, exist_ok=True)

MATRIX_PATH = os.path.join(MODELS_DIR, "user_item_matrix.npz")
USER2IDX_PATH = os.path.join(MODELS_DIR, "user2idx.pkl")
ITEM2IDX_PATH = os.path.join(MODELS_DIR, "item2idx.pkl")
IDX2ITEM_PATH = os.path.join(MODELS_DIR, "idx2item.pkl")

if os.path.exists(MATRIX_PATH) and os.path.exists(USER2IDX_PATH) and os.path.exists(ITEM2IDX_PATH) and os.path.exists(IDX2ITEM_PATH):
    print("Loading cached user-item matrix and mappings...")

    user_item_matrix = sparse.load_npz(MATRIX_PATH)
    user2idx = joblib.load(USER2IDX_PATH)
    item2idx = joblib.load(ITEM2IDX_PATH)
    idx2item = joblib.load(IDX2ITEM_PATH)

else:
    print("Building user-item matrix from scratch...")

    train_users = train_df["ncodpers"].drop_duplicates().tolist()
    user2idx = {u: i for i, u in enumerate(train_users)}
    item2idx = {p: i for i, p in enumerate(product_cols)}
    idx2item = {i: p for p, i in item2idx.items()}

    prev_cols = [f"prev_{p}" for p in product_cols]

    missing_prev_cols = [c for c in prev_cols if c not in train_df.columns]
    if missing_prev_cols:
        raise ValueError(f"Missing prev columns: {missing_prev_cols[:5]}")

    interactions = train_df[["ncodpers"] + prev_cols].melt(
        id_vars="ncodpers",
        value_vars=prev_cols,
        var_name="prev_product",
        value_name="has_product"
    )

    interactions = interactions[interactions["has_product"] == 1].copy()
    interactions["product"] = interactions["prev_product"].str.replace("prev_", "", regex=False)

    interactions = interactions.drop_duplicates(subset=["ncodpers", "product"]).copy()

    interactions["user_idx"] = interactions["ncodpers"].map(user2idx)
    interactions["item_idx"] = interactions["product"].map(item2idx)

    interactions = interactions.dropna(subset=["user_idx", "item_idx"]).copy()
    interactions["user_idx"] = interactions["user_idx"].astype(int)
    interactions["item_idx"] = interactions["item_idx"].astype(int)

    user_item_matrix = sparse.csr_matrix(
        (
            np.ones(len(interactions), dtype=np.float32),
            (interactions["user_idx"].values, interactions["item_idx"].values)
        ),
        shape=(len(user2idx), len(item2idx))
    )

    sparse.save_npz(MATRIX_PATH, user_item_matrix)
    joblib.dump(user2idx, USER2IDX_PATH)
    joblib.dump(item2idx, ITEM2IDX_PATH)
    joblib.dump(idx2item, IDX2ITEM_PATH)

    print("Saved matrix and mappings to disk.")

print("user_item_matrix shape:", user_item_matrix.shape)
print("nnz:", user_item_matrix.nnz)
print("density:", user_item_matrix.nnz / (user_item_matrix.shape[0] * user_item_matrix.shape[1]))

Building user-item matrix from scratch...
Saved matrix and mappings to disk.
user_item_matrix shape: (941815, 24)
nnz: 1456770
density: 0.06444869746181575


In [49]:
# очищаем модели

for path in [
    "models/als_model.pkl",
    "models/user_item_matrix.npz",
    "models/user2idx.pkl",
    "models/item2idx.pkl",
    "models/idx2item.pkl",
]:
    if os.path.exists(path):
        os.remove(path)
        print("Deleted:", path)

Deleted: models/als_model.pkl
Deleted: models/user_item_matrix.npz
Deleted: models/user2idx.pkl
Deleted: models/item2idx.pkl
Deleted: models/idx2item.pkl


# Обучене модели 

## Обучаем ALS

In [51]:
ALS_PATH = os.path.join(MODELS_DIR, "als_model.pkl")

if os.path.exists(ALS_PATH):
    print("Loading cached ALS model...")
    als_model = joblib.load(ALS_PATH)
else:
    print("Training ALS model from scratch...")

    als_model = AlternatingLeastSquares(
        factors=64, regularization=0.01, iterations=20, random_state=42
    )

    alpha = 20.0
    als_train_matrix = (user_item_matrix * alpha).T.tocsr()

    print("ALS train matrix shape:", als_train_matrix.shape)
    print("Expected shape: (n_items, n_users)")

    als_model.fit(user_item_matrix)

    joblib.dump(als_model, ALS_PATH)
    print("ALS model saved:", ALS_PATH)

Training ALS model from scratch...
ALS train matrix shape: (24, 941815)
Expected shape: (n_items, n_users)


100%|██████████| 20/20 [00:36<00:00,  1.80s/it]

ALS model saved: /Users/andrejmoldovan/Desktop/Магистратура РАНХИГС и Яндекс/4 семестр/Рекомендации банковских продуктов/bank-product-recs/notebooks/models/als_model.pkl


## Получаем ALS-кандидатов для пользователей validation

In [52]:
def get_existing_products_from_prev(row, product_cols):
    existing = []
    for p in product_cols:
        prev_p = f"prev_{p}"
        if prev_p in row.index and row[prev_p] == 1:
            existing.append(p)
    return existing


def recommend_als_candidates(
    user_id,
    row,
    als_model,
    user2idx,
    user_item_matrix,
    idx2item,
    product_cols,
    n_candidates=20,
):
    existing_products = set(get_existing_products_from_prev(row, product_cols))

    if user_id not in user2idx:
        return []

    user_idx = user2idx[user_id]

    user_items = user_item_matrix[user_idx]
    if user_items.nnz == 0:
        return []

    item_ids, scores = als_model.recommend(
        userid=user_idx,
        user_items=user_items,
        N=min(n_candidates + len(existing_products), len(idx2item)),
        filter_already_liked_items=False,
    )

    recs = []
    for item_idx, score in zip(item_ids, scores):
        item_idx = int(item_idx)

        if item_idx not in idx2item:
            print(f"Warning: item_idx {item_idx} not found in idx2item")
            continue

        p = idx2item[item_idx]

        if p not in existing_products:
            recs.append((p, float(score)))

        if len(recs) >= n_candidates:
            break

    return recs

In [53]:
# Проверка после обучения
sample_row = valid_df.iloc[0]
sample_user = sample_row["ncodpers"]

print("sample_user:", sample_user)
print("mapped user_idx:", user2idx.get(sample_user))
print("n_items:", len(idx2item))
print("idx2item keys sample:", list(idx2item.keys())[:10])

als_candidates = recommend_als_candidates(
    user_id=sample_user,
    row=sample_row,
    als_model=als_model,
    user2idx=user2idx,
    user_item_matrix=user_item_matrix,
    idx2item=idx2item,
    product_cols=product_cols,
    n_candidates=10,
)

print(als_candidates)

sample_user: 15889
mapped user_idx: 0
n_items: 24
idx2item keys sample: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[('ind_tjcr_fin_ult1', 1.000740647315979), ('ind_ctop_fin_ult1', 0.004976220428943634), ('ind_cno_fin_ult1', 0.004461083561182022), ('ind_nom_pens_ult1', 0.003406878560781479), ('ind_nomina_ult1', 0.0026447325944900513), ('ind_dela_fin_ult1', 0.0016498863697052002), ('ind_ahor_fin_ult1', 0.0012337062507867813), ('ind_cder_fin_ult1', 0.001048743724822998), ('ind_fond_fin_ult1', 0.0009577721357345581), ('ind_ecue_fin_ult1', 0.0008086711168289185)]


## Baseline Top Popular

### Считаем самые популярные новые продукты на train, добавляем fallback-кандидаты из Top Popular

In [58]:
# Popular products (baseline + fallback)

product_popularity = train_df[target_cols].mean()
product_popularity.index = [c.replace("target_", "") for c in product_popularity.index]

product_prev_popularity = train_df[[f"prev_{p}" for p in product_cols]].mean()
product_prev_popularity.index = [
    c.replace("prev_", "") for c in product_prev_popularity.index
]

product_popularity_df = pd.DataFrame({"product": product_cols})

product_popularity_df["popularity_new"] = product_popularity_df["product"].map(
    product_popularity
)
product_popularity_df["popularity_existing"] = product_popularity_df["product"].map(
    product_prev_popularity
)

product_popularity_df = product_popularity_df.sort_values(
    "popularity_new", ascending=False
).reset_index(drop=True)

# единый источник популярных продуктов
popular_products = product_popularity_df["product"].tolist()

# мапы для фичей
product_popularity_map = dict(
    zip(product_popularity_df["product"], product_popularity_df["popularity_new"])
)
product_prev_popularity_map = dict(
    zip(product_popularity_df["product"], product_popularity_df["popularity_existing"])
)


def get_top_popular_candidates(row, product_cols, popular_products, n_candidates=20):
    existing_products = set(get_existing_products_from_prev(row, product_cols))
    candidates = [p for p in popular_products if p not in existing_products]
    return [(p, 0.0) for p in candidates[:n_candidates]]


product_popularity_df.head()

,product,popularity_new,popularity_existing
0,ind_recibo_ult1,0.012159,0.129241
1,ind_nom_pens_ult1,0.006737,0.060056
2,ind_nomina_ult1,0.005807,0.055376
3,ind_cco_fin_ult1,0.005620,0.664876
4,ind_tjcr_fin_ult1,0.005531,0.045575


## Метрики

In [60]:
def precision_at_k(actual, predicted, k=5):
    actual_set = set(actual)
    predicted = list(dict.fromkeys(predicted[:k]))
    return len(actual_set & set(predicted)) / k if k > 0 else 0.0


def recall_at_k(actual, predicted, k=5):
    actual_set = set(actual)
    predicted = list(dict.fromkeys(predicted[:k]))
    return (
        len(actual_set & set(predicted)) / len(actual_set)
        if len(actual_set) > 0
        else 0.0
    )


def apk(actual, predicted, k=5):
    actual_set = set(actual)
    predicted = predicted[:k]

    score = 0.0
    hits = 0.0
    seen = set()

    for i, p in enumerate(predicted):
        if p in actual_set and p not in seen:
            hits += 1.0
            score += hits / (i + 1.0)
        seen.add(p)

    return score / min(len(actual_set), k) if len(actual_set) > 0 else 0.0


def mean_metrics(actual_list, predicted_list, k=5):
    precisions = [precision_at_k(a, p, k) for a, p in zip(actual_list, predicted_list)]
    recalls = [recall_at_k(a, p, k) for a, p in zip(actual_list, predicted_list)]
    maps = [apk(a, p, k) for a, p in zip(actual_list, predicted_list)]

    return {
        f"precision_at_{k}": float(np.mean(precisions)),
        f"recall_at_{k}": float(np.mean(recalls)),
        f"map_at_{k}": float(np.mean(maps)),
    }

## Подготовка фактических ответов для valid

In [61]:
# Actual для validation
def extract_actual_products(row, target_cols):
    return [c.replace("target_", "") for c in target_cols if row[c] == 1]


def get_existing_products_from_prev(row, product_cols):
    return [p for p in product_cols if row.get(f"prev_{p}", 0) == 1]


def top_popular_recommendations(row, top_popular_products, product_cols, k=5):
    existing_products = set(get_existing_products_from_prev(row, product_cols))
    preds = [p for p in top_popular_products if p not in existing_products]
    return preds[:k]


valid_actual = (
    valid_df[target_cols]
    .apply(lambda row: extract_actual_products(row, target_cols), axis=1)
    .tolist()
)

print("valid rows:", len(valid_actual))
print("non-empty actual:", sum(len(x) > 0 for x in valid_actual))
print(
    "share non-empty actual:",
    round(sum(len(x) > 0 for x in valid_actual) / len(valid_actual), 4),
)

valid rows: 926760
non-empty actual: 27916
share non-empty actual: 0.0301


## Baseline Top Popular

In [63]:
K = 5

popular_targets = train_df[target_cols].sum().sort_values(ascending=False)
top_popular_products = [
    c.replace("target_", "") for c in popular_targets.index.tolist()
]

valid_pred_top_pop = [
    top_popular_recommendations(row, top_popular_products, product_cols, k=K)
    for _, row in valid_df.iterrows()
]

baseline_metrics = mean_metrics(valid_actual, valid_pred_top_pop, k=K)
print("Baseline metrics (all):", baseline_metrics)

mask_non_empty = [len(x) > 0 for x in valid_actual]
valid_actual_pos = [a for a, m in zip(valid_actual, mask_non_empty) if m]
valid_pred_top_pop_pos = [p for p, m in zip(valid_pred_top_pop, mask_non_empty) if m]

print(
    "Baseline metrics (positive rows only):",
    mean_metrics(valid_actual_pos, valid_pred_top_pop_pos, k=K),
)

Baseline metrics (all): {'precision_at_5': 0.0069791531788165224, 'recall_at_5': 0.02664050023738616, 'map_at_5': 0.018953044297696633}
Baseline metrics (positive rows only): {'precision_at_5': 0.23169508525576735, 'recall_at_5': 0.8844157472417251, 'map_at_5': 0.6292063094044037}


## Обучение LightGBM one-vs-rest

### По одной модели на каждый продукт.

In [64]:
models = {}
valid_scores = pd.DataFrame(index=valid_df.index)

params = {
    "objective": "binary",
    "metric": ["binary_logloss", "auc"],
    "learning_rate": 0.05,
    "num_leaves": 31,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "verbosity": -1,
    "seed": 42,
}

MIN_POSITIVES = 300
MODEL_DIR = "models/ovr"
os.makedirs(MODEL_DIR, exist_ok=True)

for i, target_col in enumerate(target_cols, start=1):
    y_train = train_df[target_col].astype(int)
    y_valid = valid_df[target_col].astype(int)

    n_pos_train = int(y_train.sum())
    n_pos_valid = int(y_valid.sum())

    print(
        f"[{i}/{len(target_cols)}] {target_col} | "
        f"train positives={n_pos_train}, valid positives={n_pos_valid}"
    )

    if n_pos_train < MIN_POSITIVES:
        print(f"  -> skip: too few positives in train (< {MIN_POSITIVES})")
        valid_scores[target_col] = 0.0
        continue

    if y_train.nunique() < 2:
        print("  -> skip: only one class in train")
        valid_scores[target_col] = 0.0
        continue

    train_dataset = lgb.Dataset(
        X_train, label=y_train, categorical_feature=cat_features, free_raw_data=False
    )

    valid_dataset = lgb.Dataset(
        X_valid, label=y_valid, categorical_feature=cat_features, free_raw_data=False
    )

    pos_weight = (len(y_train) - y_train.sum()) / max(y_train.sum(), 1)

    params_local = params.copy()
    params_local["scale_pos_weight"] = pos_weight

    model = lgb.train(
        params_local,
        train_dataset,
        valid_sets=[valid_dataset],
        num_boost_round=300,
        callbacks=[lgb.early_stopping(30), lgb.log_evaluation(50)],
    )

    models[target_col] = model
    valid_scores[target_col] = model.predict(
        X_valid, num_iteration=model.best_iteration
    )

    joblib.dump(model, os.path.join(MODEL_DIR, f"{target_col}.pkl"))

print(f"\nОбучено моделей: {len(models)} из {len(target_cols)}")
print("Форма valid_scores:", valid_scores.shape)

[1/24] target_ind_ahor_fin_ult1 | train positives=1, valid positives=1
  -> skip: too few positives in train (< 300)
[2/24] target_ind_aval_fin_ult1 | train positives=4, valid positives=0
  -> skip: too few positives in train (< 300)
[3/24] target_ind_cco_fin_ult1 | train positives=66119, valid positives=3878
Training until validation scores don't improve for 30 rounds
[50]	valid_0's binary_logloss: 0.0614266	valid_0's auc: 0.997062
[100]	valid_0's binary_logloss: 0.0547458	valid_0's auc: 0.997434
[150]	valid_0's binary_logloss: 0.0505745	valid_0's auc: 0.997594
[200]	valid_0's binary_logloss: 0.0474387	valid_0's auc: 0.997704
[250]	valid_0's binary_logloss: 0.0451928	valid_0's auc: 0.99777
[300]	valid_0's binary_logloss: 0.0434988	valid_0's auc: 0.997835
Did not meet early stopping. Best iteration is:
[300]	valid_0's binary_logloss: 0.0434988	valid_0's auc: 0.997835
[4/24] target_ind_cder_fin_ult1 | train positives=131, valid positives=5
  -> skip: too few positives in train (< 300)
[

## Формирование рекомендаций модели

### Важно не рекомендовать уже имеющиеся у клиента продукты.

In [ ]:
def get_model_recommendations(
    valid_df,
    score_df,
    product_cols,
    prev_product_cols,
    top_k=5,
    top_popular_products=None
):
    recommendations = []

    prev_to_product = {f"prev_{p}": p for p in product_cols}

    for idx in valid_df.index:
        # ИСКЛЮЧАЕМ только продукты, которые уже были до текущего месяца
        existing_products = set(
            prev_to_product[col]
            for col in prev_product_cols
            if valid_df.loc[idx, col] == 1
        )

        scores = {
            target_col.replace("target_", ""): score_df.loc[idx, target_col]
            for target_col in score_df.columns
        }

        filtered_scores = {
            p: s for p, s in scores.items() if p not in existing_products
        }

        ranked = sorted(filtered_scores.items(), key=lambda x: x[1], reverse=True)
        recs = [p for p, _ in ranked[:top_k]]

        if top_popular_products is not None and len(recs) < top_k:
            for p in top_popular_products:
                if p not in recs and p not in existing_products:
                    recs.append(p)
                if len(recs) == top_k:
                    break

        recommendations.append(recs)

    return recommendations

## Оценка LightGBM

In [ ]:
valid_pred_lgb = get_model_recommendations(
    valid_df=valid_df,
    score_df=valid_scores,
    product_cols=product_cols,
    prev_product_cols=prev_product_cols,
    top_k=K,
    top_popular_products=top_popular_products
)

lgb_metrics = mean_metrics(valid_actual, valid_pred_lgb, k=K)
print("LightGBM metrics (all):", lgb_metrics)

valid_pred_lgb_pos = [p for p, m in zip(valid_pred_lgb, mask_non_empty) if m]
print("LightGBM metrics (positive rows only):", mean_metrics(valid_actual_pos, valid_pred_lgb_pos, k=K))

LightGBM metrics (all): {'precision_at_5': 0.007249125987310632, 'recall_at_5': 0.028280964507171928, 'map_at_5': 0.01908188533768781}
LightGBM metrics (positive rows only): {'precision_at_5': 0.24065768734775758, 'recall_at_5': 0.9388761522663228, 'map_at_5': 0.6334835956281543}


## Сравнение моделей

In [ ]:
# функция для табличного стравнения моделей
def add_model_result(
    results_df: pd.DataFrame,
    model_name: str,
    metrics: dict,
    extra_info: dict | None = None
) -> pd.DataFrame:
    """
    Добавляет результаты модели в таблицу сравнения.

    Parameters
    ----------
    results_df : pd.DataFrame
        Текущая таблица результатов.
    model_name : str
        Название модели.
    metrics : dict
        Словарь метрик, например:
        {"precision_at_5": 0.01, "recall_at_5": 0.03, "map_at_5": 0.02}
    extra_info : dict | None
        Дополнительная информация, например:
        {"split": "valid", "notes": "with prev features"}

    Returns
    -------
    pd.DataFrame
        Обновлённая таблица результатов.
    """
    row = {"model": model_name}
    row.update(metrics)

    if extra_info is not None:
        row.update(extra_info)

    row_df = pd.DataFrame([row])

    if results_df is None or results_df.empty:
        return row_df

    return pd.concat([results_df, row_df], ignore_index=True)

In [ ]:
comparison_df = pd.DataFrame()

comparison_df = add_model_result(
    comparison_df,
    model_name="Top Popular",
    metrics=baseline_metrics,
    extra_info={"evaluation": "all_rows"}
)

comparison_df = add_model_result(
    comparison_df,
    model_name="LightGBM",
    metrics=lgb_metrics,
    extra_info={"evaluation": "all_rows"}
)

comparison_df = add_model_result(
    comparison_df,
    model_name="Top Popular",
    metrics=mean_metrics(valid_actual_pos, valid_pred_top_pop_pos, k=5),
    extra_info={"evaluation": "positive_rows_only"}
)

comparison_df = add_model_result(
    comparison_df,
    model_name="LightGBM",
    metrics=mean_metrics(valid_actual_pos, valid_pred_lgb_pos, k=5),
    extra_info={"evaluation": "positive_rows_only"}
)

comparison_df = comparison_df.sort_values(["evaluation", "map_at_5"], ascending=[True, False]).reset_index(drop=True)
display(comparison_df)

,model,precision_at_5,recall_at_5,map_at_5,evaluation
0,LightGBM,0,0,0,all_rows
1,Top Popular,0,0,0,all_rows
2,LightGBM,0,1,1,positive_rows_only
3,Top Popular,0,1,1,positive_rows_only


## Логирование в MLflow

In [ ]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5001")
mlflow.set_experiment("bank_product_recommendation")

with mlflow.start_run(run_name="lightgbm_one_vs_rest_v1"):

    # параметры
    mlflow.log_param("model_type", "lightgbm_one_vs_rest")
    mlflow.log_param("top_k", K)
    mlflow.log_param("num_features", len(feature_cols))
    mlflow.log_param("num_products", len(product_cols))

    # baseline метрики
    for metric_name, metric_value in baseline_metrics.items():
        mlflow.log_metric(f"baseline_{metric_name}", float(metric_value))

    # модельные метрики
    for metric_name, metric_value in lgb_metrics.items():
        mlflow.log_metric(metric_name, float(metric_value))

print("MLflow logging completed")

🏃 View run lightgbm_one_vs_rest_v1 at: http://127.0.0.1:5001/#/experiments/1/runs/5fb897cf93254efba00d9c0d716edcae
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/1
MLflow logging completed


## Сохранение модели

In [ ]:
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
os.makedirs(MODELS_DIR, exist_ok=True)

# pkl
joblib.dump(models, os.path.join(MODELS_DIR, "lgbm_one_vs_rest.pkl"))

# bin
for target_col, model in models.items():
    model.save_model(os.path.join(MODELS_DIR, f"{target_col}.bin"))

print("Модели сохранены")

Модели сохранены


In [ ]:
print("TRAIN target sum:", train_df[target_cols].sum().sum())
print("VALID target sum:", valid_df[target_cols].sum().sum())

valid_actual = valid_df[target_cols].apply(
    lambda row: [c.replace("target_", "") for c in target_cols if row[c] == 1],
    axis=1
).tolist()

num_non_empty_actual = sum(len(x) > 0 for x in valid_actual)
print("Non-empty actual in valid:", num_non_empty_actual)
print("Share:", round(num_non_empty_actual / len(valid_actual), 4))

print("\nTop valid targets:")
display(valid_df[target_cols].sum().sort_values(ascending=False).head(10))

print("\nScore stats:")
display(valid_scores.describe())

print("\nExamples:")
for i in range(5):
    print("ACTUAL:", valid_actual[i])
    print("PRED  :", valid_pred_lgb[i])
    print("-" * 40)

TRAIN target sum: 527422
VALID target sum: 35888
Non-empty actual in valid: 27916
Share: 0.0301

Top valid targets:


target_ind_recibo_ult1      10163
target_ind_nom_pens_ult1     5513
target_ind_nomina_ult1       5488
target_ind_tjcr_fin_ult1     4248
target_ind_cco_fin_ult1      3878
target_ind_ecue_fin_ult1     2709
target_ind_cno_fin_ult1      2347
target_ind_ctma_fin_ult1      531
target_ind_reca_fin_ult1      279
target_ind_ctop_fin_ult1      226
dtype: uint64


Score stats:


,target_ind_ahor_fin_ult1,target_ind_aval_fin_ult1,target_ind_cco_fin_ult1,target_ind_cder_fin_ult1,target_ind_cno_fin_ult1,target_ind_ctju_fin_ult1,target_ind_ctma_fin_ult1,target_ind_ctop_fin_ult1,target_ind_ctpp_fin_ult1,target_ind_deco_fin_ult1,target_ind_deme_fin_ult1,target_ind_dela_fin_ult1,target_ind_ecue_fin_ult1,target_ind_fond_fin_ult1,target_ind_hip_fin_ult1,target_ind_plan_fin_ult1,target_ind_pres_fin_ult1,target_ind_reca_fin_ult1,target_ind_tjcr_fin_ult1,target_ind_valo_fin_ult1,target_ind_viv_fin_ult1,target_ind_nomina_ult1,target_ind_nom_pens_ult1,target_ind_recibo_ult1
count,"926,760","926,760","926,760","926,760","926,760","926,760","926,760","926,760","926,760","926,760","926,760","926,760","926,760","926,760","926,760","926,760","926,760","926,760","926,760","926,760","926,760","926,760","926,760","926,760"
mean,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
std,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
min,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
25%,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
50%,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
75%,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,1
max,0,0,1,0,1,1,1,1,1,1,0,1,1,1,0,1,0,1,1,1,0,1,1,1



Examples:
ACTUAL: ['ind_tjcr_fin_ult1']
PRED  : ['ind_ctju_fin_ult1', 'ind_recibo_ult1', 'ind_fond_fin_ult1', 'ind_tjcr_fin_ult1', 'ind_ctop_fin_ult1']
----------------------------------------
ACTUAL: []
PRED  : ['ind_ctju_fin_ult1', 'ind_valo_fin_ult1', 'ind_fond_fin_ult1', 'ind_ctop_fin_ult1', 'ind_cco_fin_ult1']
----------------------------------------
ACTUAL: []
PRED  : ['ind_ctju_fin_ult1', 'ind_plan_fin_ult1', 'ind_fond_fin_ult1', 'ind_ctop_fin_ult1', 'ind_nom_pens_ult1']
----------------------------------------
ACTUAL: []
PRED  : ['ind_ctju_fin_ult1', 'ind_plan_fin_ult1', 'ind_cco_fin_ult1', 'ind_fond_fin_ult1', 'ind_tjcr_fin_ult1']
----------------------------------------
ACTUAL: []
PRED  : ['ind_ctju_fin_ult1', 'ind_plan_fin_ult1', 'ind_cno_fin_ult1', 'ind_fond_fin_ult1', 'ind_ctop_fin_ult1']
----------------------------------------


In [ ]:
# 1. actual
valid_actual = valid_df[target_cols].apply(
    lambda row: [c.replace("target_", "") for c in target_cols if row[c] == 1],
    axis=1
).tolist()

# 2. метрики по всем
print("Baseline metrics (all):", mean_metrics(valid_actual, valid_pred_top_pop, k=5))
print("LightGBM metrics (all):", mean_metrics(valid_actual, valid_pred_lgb, k=5))

# 3. метрики только по строкам с положительными событиями
mask_non_empty = [len(x) > 0 for x in valid_actual]

valid_actual_pos = [a for a, m in zip(valid_actual, mask_non_empty) if m]
valid_pred_top_pop_pos = [p for p, m in zip(valid_pred_top_pop, mask_non_empty) if m]
valid_pred_lgb_pos = [p for p, m in zip(valid_pred_lgb, mask_non_empty) if m]

print("Baseline metrics (positive rows only):", mean_metrics(valid_actual_pos, valid_pred_top_pop_pos, k=5))
print("LightGBM metrics (positive rows only):", mean_metrics(valid_actual_pos, valid_pred_lgb_pos, k=5))

# 4. сколько таких строк
print("Positive rows in valid:", sum(mask_non_empty), "out of", len(mask_non_empty))

Baseline metrics (all): {'precision_at_5': 0.00632094609176054, 'recall_at_5': 0.024207364006502932, 'map_at_5': 0.015614864629941346}
LightGBM metrics (all): {'precision_at_5': 0.007249125987310632, 'recall_at_5': 0.028280964507171928, 'map_at_5': 0.01908188533768781}
Baseline metrics (positive rows only): {'precision_at_5': 0.2098438171657831, 'recall_at_5': 0.8036400869274489, 'map_at_5': 0.5183848669022941}
LightGBM metrics (positive rows only): {'precision_at_5': 0.24065768734775758, 'recall_at_5': 0.9388761522663228, 'map_at_5': 0.6334835956281543}
Positive rows in valid: 27916 out of 926760
